# ComplaintIQ - live scoring demo

Score a single consumer complaint the way a reviewer's queue would: paste or pick a narrative, and see where it lands in the review queue (top-1/5/10%) and which words drove the score.

The model is the narrative TF-IDF + logistic regression from notebook 09 (PR-AUC ~0.27 on the recent-months test window). It ranks complaints by relief risk. Because it's trained with `class_weight=balanced` for ranking, the raw score isn't a calibrated probability, so the demo reports a **queue band**, not a percentage.

In [ ]:
# Load the trained model. Prefer the MLflow registry (@champion); fall back to
# the joblib dump so the live demo works even if the registry is unavailable.
import json
from pathlib import Path

import numpy as np


def _find_repo() -> Path:
    """Locate the repo root (the dir containing models/) robustly.

    Search, in order: (1) cwd and its parents - covers local runs from
    notebooks/ or the repo root; (2) the notebook's own workspace path and its
    parents - covers Databricks, where cwd is usually / or a temp dir, NOT where
    the notebook lives. The notebook path comes from the injected dbutils
    context. First location that contains models/relief_pipeline.joblib wins.
    """
    candidates = [Path.cwd(), *Path.cwd().parents]
    try:
        nb_path = (
            dbutils.notebook.entry_point.getDbutils()  # noqa: F821 - Databricks global
            .notebook().getContext().notebookPath().get()
        )
        nb = Path("/Workspace") / nb_path.lstrip("/")
        candidates += [nb, *nb.parents]
    except Exception:
        pass  # Not on Databricks (no dbutils); cwd search above is enough.
    for p in candidates:
        if (p / "models" / "relief_pipeline.joblib").exists():
            return p
    # Last resort: keep the old heuristic so the error message is sensible.
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


REPO = _find_repo()

pipe = None
try:
    import mlflow

    mlflow.set_tracking_uri(f"sqlite:///{REPO / 'mlflow.db'}")
    pipe = mlflow.sklearn.load_model("models:/complaintiq_relief@champion")
    print("loaded model from MLflow registry: complaintiq_relief@champion")
except Exception as e:
    # Registry unavailable (e.g. on Databricks, where the sqlite store isn't
    # synced); the joblib fallback is git-tracked so it travels with the repo.
    import joblib

    pipe = joblib.load(REPO / "models" / "relief_pipeline.joblib")
    print(f"MLflow unavailable ({type(e).__name__}); loaded joblib fallback from {REPO}")

# Cross-version compat shim: the model is pickled with scikit-learn 1.9.0, which
# dropped LogisticRegression.multi_class. Databricks serverless notebooks run an
# older sklearn whose predict_proba still reads that attribute, so the unpickled
# estimator is missing it and scoring raises AttributeError. Restore it if
# absent: for a binary model "deprecated" routes to the correct sigmoid path,
# and it's ignored by newer sklearn. Cheaper and more demo-safe than a %pip pin.
_clf = pipe.named_steps["clf"]
if not hasattr(_clf, "multi_class"):
    _clf.multi_class = "deprecated"

art = json.loads((REPO / "models" / "demo_artifacts.json").read_text())
THRESH = art["queue_thresholds"]
print("queue thresholds:", THRESH)
print("test base rate:", art["test_base_rate"])

In [ ]:
from typing import Any

# Scoring helpers.
vec = pipe.named_steps["tfidf"]
clf = pipe.named_steps["clf"]
feature_names = np.array(vec.get_feature_names_out())
coef = clf.coef_[0]


def queue_band(score: float) -> str:
    """Map relief-risk score to interpretable queue band for reviewer triage."""
    if score >= THRESH["top01"]:
        return "TOP 1% - specialist review (highest relief risk)"
    if score >= THRESH["top05"]:
        return "TOP 5% - priority review"
    if score >= THRESH["top10"]:
        return "TOP 10% - elevated review"
    return "below top 10% - standard queue"


def top_drivers(text: str, k: int = 8) -> list[tuple[str, float]]:
    """Return top-k words driving the score up (tfidf weight * logreg coef > 0 only)."""
    # Isolate indices where text has nonzero tfidf; element-wise multiply by coef.
    tfidf_vec = vec.transform([text])
    present_indices = tfidf_vec.indices
    contributions = tfidf_vec.data * coef[present_indices]
    descending_order = np.argsort(contributions)[::-1]
    # Keep only positive contributors; map back to feature names.
    top_k = [
        (feature_names[present_indices[i]], float(contributions[i]))
        for i in descending_order[:k]
        if contributions[i] > 0
    ]
    return top_k


def score_text(text: str) -> dict[str, Any]:
    """Score narrative, returning calibration-free rank score, band, and top drivers."""
    score = float(pipe.predict_proba([text])[0, 1])
    return {"score": score, "band": queue_band(score), "drivers": top_drivers(text)}

In [ ]:
# Example complaints to demo with (redacted CFPB-style narratives), ordered from
# highest to lowest relief risk so the demo shows the queue spread.
EXAMPLES = {
    "Billing overcharge / duplicate (high risk)": "I was charged twice for the same payment and the company will not reverse "
    "the duplicate charge or refund me the money I am owed.",
    "Unauthorized charge / fraud (high risk)": "There were unauthorized charges on my account totaling $450 that I did "
    "not make. I reported the fraud to my bank but they refused to refund the "
    "money and closed my claim without explanation.",
    "Debt collection harassment (rarely monetary)": "A debt collector keeps calling me multiple times a day about a debt that "
    "is not mine. They threatened legal action and refused to send validation "
    "of the debt when I asked.",
    "General mortgage question (low risk)": "I wanted to ask a question about my mortgage statement and how escrow is "
    "calculated for the coming year. Everything is fine, just looking for an "
    "explanation of the numbers.",
}

## Score a complaint
Pick an example or paste your own, then click **Score complaint**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

example_dropdown = widgets.Dropdown(
    options=["(custom)"] + list(EXAMPLES),
    value="(custom)",
    description="Example:",
    layout=widgets.Layout(width="60%"),
)
narrative_input = widgets.Textarea(
    placeholder="Paste a complaint narrative...",
    layout=widgets.Layout(width="90%", height="140px"),
)
score_button = widgets.Button(description="Score complaint", button_style="primary")
result_output = widgets.Output()


def _on_example_select(change: dict[str, Any]) -> None:
    """Populate narrative_input when user picks an example."""
    selected_key = change["new"]
    if selected_key != "(custom)":
        narrative_input.value = EXAMPLES[selected_key]


def _on_score_click(_: Any) -> None:
    """Score the narrative and display band + driver words."""
    with result_output:
        clear_output()
        narrative = narrative_input.value.strip()
        if not narrative:
            print("Enter or pick a complaint first.")
            return
        result = score_text(narrative)
        driver_words = (
            ", ".join(f"`{word}`" for word, _ in result["drivers"])
            or "(no strong positive drivers)"
        )
        display(
            Markdown(
                f"### Relief-risk score: {result['score']:.3f}\n"
                f"**Queue placement:** {result['band']}\n\n"
                f"**Top driver words:** {driver_words}"
            )
        )


example_dropdown.observe(_on_example_select, names="value")
score_button.on_click(_on_score_click)
display(example_dropdown, narrative_input, score_button, result_output)

## How to read this
- **Queue placement** is what the product delivers: reviewers work the top of the ranked queue first. A complaint in the **top 1%** band is where the model reaches ~59x lift - roughly 1 in 5 of those is a genuine relief case, versus ~0.34% at random.
- **Driver words** are the terms pushing the score up (TF-IDF weight x model coefficient), so a reviewer sees *why* it was flagged, not just that it was.
- The score ranks; it is not a calibrated probability (the model is class-weighted for ranking). That's why we show a band, not a percentage.